**Universidad Internacional de La Rioja (UNIR) - Máster Universitario en Inteligencia Artificial - Procesamiento del Lenguaje Natural** 

***
Datos del alumno (Nombre y Apellidos): Andrés Hurtado Morón

Fecha: 18/10/2026
***

<span style="font-size: 20pt; font-weight: bold; color: #0098cd;">Trabajo: Caracterización de textos</span>

**Objetivos** 

Con esta actividad se tratará de que el alumno se familiarice con el manejo de la librería spacy, así como con los conceptos básicos de manejo de las técnicas NER

**Descripción**

En esta actividad debes procesar de forma automática un texto en lenguaje natural para detectar características básicas en el mismo, y para identificar y etiquetar las ocurrencias de conceptos como localización, moneda, empresas, etc.

En la primera parte del ejercicio se proporciona un código fuente a través del cual se lee un archivo de texto y se realiza un preprocesado del mismo. En esta parte el alumno tan sólo debe ejecutar y entender el código proporcionado.

En la segunda parte del ejercicio se plantean una serie de preguntas que deben ser respondidas por el alumno. Cada pregunta deberá responderse con un fragmento de código fuente que esté acompañado de la explicación correspondiente. Para elaborar el código solicitado, el alumno deberá visitar la documentación de la librería spacy, cuyos enlaces se proporcionarán donde corresponda.

# Parte 1: carga y preprocesamiento del texto a analizar

Observa las diferentes librerías que se están importando.

In [2]:
import pathlib
import spacy
import pandas as pd
from spacy import displacy
import csv
import es_core_news_md
from pathlib import Path

El siguiente código simplemente carga y preprocesa el texto. Para ello, lo primero que hace es cargar un modelo de lenguaje previamente entrenado. En este caso, se utiliza <i>es_core_news_md</i>: 

https://spacy.io/models/es#es_core_news_md


In [3]:
nlp = es_core_news_md.load()

El objeto <i>nlp</i> permite utilizar el modelo de lenguaje cargado, de forma que se puede procesar un texto y obtenerlo en su versión preprocesada. Así, nos permite realizar las diferentes tareas. En este caso, vamos a utilizar el pipeline para hacer un preprocesamiento básico, que consiste en tokenizar el texto.

In [4]:
filename = "02Dataset_anonimizado.csv"
lines_number  = 10000
SEMILLA = 42

data = pd.read_csv(
    filename,
    delimiter=';',
    encoding='latin-1',
)

registros_total = len(data)
print(registros_total)

575029


/tmp/ipykernel_4603/2997666523.py:5: DtypeWarning: Columns (6: TIPO DE ODIO, 7: TONO HUMORISTICO, 8: MODIFICADOR) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(


El código anterior carga el archivo CSV (opcionalmente con un límite de líneas a leer) y genera la variable <i>data</i>, que contiene un Dataframe (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) con los datos leídos del CSV.

Te vendrá bien conocer la siguiente documentación:
<ul>
    <li>https://spacy.io/api/doc</li>
    <li>https://spacy.io/api/token</li>
    <li>https://spacy.io/api/morphology#morphanalysis</li>
</ul>

### Playground

Utiliza este espacio para hacer pruebas y ensayos con las variables generadas con el código previo. A modo de ejemplo, se ofrece código que realiza las siguientes tareas: 


- leer un número dado de líneas del Dataframe y generar dos listas con los valores (se pueden leer directamente del DataFrame, se muestra el ejemplo como una opción más)
- procesar el texto de cada comentario


Para procesarlo, hay utilizar el objeto <i>nlp</i> y así obtener objetos de la clase <i>Doc</i> (https://spacy.io/api/doc)

Visita la documentación de dicha clase y experimenta probando las diferentes funciones y atributos 

Primero, filtramos entre registros que sean comentarios o no

In [5]:
comentarios = data[data["TIPO DE MENSAJE"] == "COMENTARIO"].copy()
comentarios = comentarios[comentarios["INTENSIDAD"].notna()].copy()

# Guardamos el total ANTES de muestrear: comentarios sigue siendo la poblacion completa.
n_comentarios = len(comentarios)

#Ahora data es una muestra de 10.000 comentarios
data = comentarios.sample(n=lines_number, random_state=SEMILLA).reset_index(drop=True)

Comprobamos ejemplos del dataset para ver si tenemos caracteres extraños:

In [6]:
data.head(10)

,MEDIO,SOPORTE,URL,TIPO DE MENSAJE,CONTENIDO A ANALIZAR,INTENSIDAD,TIPO DE ODIO,TONO HUMORISTICO,MODIFICADOR,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,ABC,TWITTER,URL_778175f4,COMENTARIO,hay mucho chuleta suelto...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20MIN,WEB,URL_fbbca4b5,COMENTARIO,.........cambia de novio.,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,La Vanguardia,WEB,URL_d034375f,COMENTARIO,[USUARIO_OFUSCADO]Â·í±ÃÂ [USUARIO_OFUSCADO] l...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABC,WEB,URL_e3d5e1d3,COMENTARIO,al psoe no le interesa la derrota del separati...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABC,TWITTER,URL_dec03b12,COMENTARIO,[USUARIO_OFUSCADO] somos unos mierdas irrespon...,4.0,Otros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,EL PAÍS,TWITTER,URL_439e18e9,COMENTARIO,ÃÂ¿y cuántas personas han sido ya vacunadas e...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ABC,TWITTER,URL_7c8cc2c7,COMENTARIO,"7 la ley de jehová es perfecta, que convierte ...",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,EL MUNDO,WEB,URL_f2a63fe2,COMENTARIO,[USUARIO_OFUSCADO] #57 entonces por qué dios t...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,EL MUNDO,WEB,URL_4cefcbe5,COMENTARIO,"[USUARIO_OFUSCADO] #2 no se sulfure, que no ti...",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,EL MUNDO,TWITTER,URL_32c7fe54,COMENTARIO,"ÃÂ¿será acaso el uno de los miembros del ""com...",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Limpieza del mojibake

El fichero se lee con `encoding='latin-1'` porque no es UTF-8 válido (los bytes `0xED` de `í±`, `í¼`, `í§`... no lo son) y con `utf-8` estricto lanza `UnicodeDecodeError`. Usar `ignore` o `replace` perdería tildes y `ñ`, que son información lingüística.

El texto tiene **varias capas** de doble codificación (`Â¿` aparece como `Ã\x82Â¿`, la `ñ` como `í±`), así que una tabla fija de pares no llega a todo. Generamos la tabla de reemplazos a partir de las reglas de codificación, la ordenamos de claves largas a cortas y la aplicamos repetidamente hasta que el texto deja de cambiar.

In [14]:
replacements = {}

# Pares "Âx" y "Ãx" (x = byte de continuación) -> carácter real.
for b in range(0x80, 0xC0):
    replacements[chr(0xC2) + chr(b)] = chr(b)
    replacements[chr(0xC3) + chr(b)] = chr(0xC0 | (b & 0x3F))

# En este corpus, "í" y "ã" también aparecen como primer byte de letras acentuadas.
for b in range(0xA0, 0xC0):
    replacements[chr(0xED) + chr(b)] = chr(0xC0 | (b & 0x3F))
    replacements[chr(0xE3) + chr(b)] = chr(0xC0 | (b & 0x3F))

# Signos de 3 bytes que las reglas anteriores dejan a medias.
replacements.update({
    "â\x80\x9c": "“", "â\x80\x9d": "”", "â\x80\x98": "‘", "â\x80\x99": "’",
    "â\x80\x93": "–", "â\x80\x94": "—", "â\x80\x95": "―", "â\x80\xa6": "…",
    "â\x82¬": "€", "â\x80¢": "•", "â€¦" : "…"
})

# Claves largas primero para no romper secuencias largas.
replacements = dict(sorted(replacements.items(), key=lambda kv: -len(kv[0])))

In [15]:
def fix_mojibake(s):
    if not isinstance(s, str):
        return ""
    prev = None
    while prev != s:
        prev = s
        for bad, good in replacements.items():
            s = s.replace(bad, good)
    return s

In [13]:
#LLAMAMOS A NUESTRA FUNCION PARA LIMPIAR EL TEXTO DE LOS COMENTARIOS
data["CONTENIDO A ANALIZAR"] = [fix_mojibake(t) for t in data["CONTENIDO A ANALIZAR"].values]

print("restos de 'Ã':", int(data["CONTENIDO A ANALIZAR"].str.count("Ã").sum()))
print("restos de '�':", int(data["CONTENIDO A ANALIZAR"].str.count("�").sum()))
print(data["CONTENIDO A ANALIZAR"].iloc[2][:120])

restos de 'Ã': 0
restos de '�': 0
[USUARIO_OFUSCADO]·ñ [USUARIO_OFUSCADO] los del parlament no eran 2.3 millones...


In [16]:
import re

patron = re.compile(r"�|Ã|Â|â€|â€™|â€œ|â€“|â€”")

columna = "CONTENIDO A ANALIZAR"

sospechosos = data[
    data[columna].astype("string").str.contains(patron, na=False)
]

print("Comentarios sospechosos:", len(sospechosos))
print(sospechosos[columna].head(5).to_string(index=False))

Comentarios sospechosos: 2
deja de meter las narices en el dormitorio de o...
la corte costituzionale italiana (sentencia del...


In [17]:
data.head(5)

,MEDIO,SOPORTE,URL,TIPO DE MENSAJE,CONTENIDO A ANALIZAR,INTENSIDAD,TIPO DE ODIO,TONO HUMORISTICO,MODIFICADOR,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,ABC,TWITTER,URL_778175f4,COMENTARIO,hay mucho chuleta suelto...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20MIN,WEB,URL_fbbca4b5,COMENTARIO,.........cambia de novio.,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,La Vanguardia,WEB,URL_d034375f,COMENTARIO,[USUARIO_OFUSCADO]·ñ [USUARIO_OFUSCADO] los de...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABC,WEB,URL_e3d5e1d3,COMENTARIO,al psoe no le interesa la derrota del separati...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABC,TWITTER,URL_dec03b12,COMENTARIO,[USUARIO_OFUSCADO] somos unos mierdas irrespon...,4.0,Otros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
doc = []
value = []

#con el bucle, generamos sendas listas con los comentarios ya parseados y con el valor de intensidad
for i in range(0, lines_number):
    
    #en un primer paso se parsea el comentario. En el segundo paso se añade el objeto a la lista
    tmp_doc = nlp(data["CONTENIDO A ANALIZAR"][i])
    doc.append(tmp_doc)
    
    #en un primer paso extrae el valor. En el segundo paso se añade el valor a la lista
    tmp_value = data["INTENSIDAD"][i]
    value.append(tmp_value)



In [19]:
#ejemplo de cómo recorrer un comentario palabra por palabra    
for token in doc[2]:
    print(token)

[
USUARIO_OFUSCADO]·ñ
 
[
USUARIO_OFUSCADO
]
los
del
parlament
no
eran
2.3
millones
...


In [20]:
#visualización de las lista value y doc, que contienen los valores de intensidad y los comentarios parseados, respectivamente
print(value[:5])
print(doc[:5])

[np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(4.0)]
[hay mucho chuleta suelto..., .........cambia de novio., [USUARIO_OFUSCADO]·ñ [USUARIO_OFUSCADO] los del parlament no eran 2.3 millones..., al psoe no le interesa la derrota del separatismo

 

no falla, siempre que los separatistas sufren un grave fracaso que les puede conducir a la derrota final, sale el psoe a su rescate. a trancas y barrancas, con sánchez apoyando a regañadientes, rajoy aplicó el artículo 155 tras el golpe de estado de los separatistas, y posteriormente vino el juicio a los golpistas y la condena por sedición. fue un evidente triunfo del estado constitucional español y una clara derrota del separatismo. pues tuvo que llegar el psoe al gobierno, para darle la vuelta a la tortilla y transformar esa victoria de la democracia española en derrota. no solo por los indultos, sino por un proceso de dialogo posterior que consistirá en dar más cesiones a los separatistas, para que el próxim

He creado un diccionario para tener agrupada, por cada comentario, toda la información que necesito en las siguientes preguntas: el texto, su número de palabras (criterio is_alpha, el mismo de la Pregunta 2) y su número de oraciones (len(list(d.sents))). Recorro doc y value a la vez con zip, de modo que cada Doc queda emparejado con su INTENSIDAD, y guardo cada comentario como un diccionario dentro de comentarios_corpus. A la vez lo clasifico en odio (intensidad > 0) o no_odio (intensidad == 0); así, para obtener cada promedio solo hay que sumar num_palabras (o num_oraciones) del grupo y dividir entre el número de comentarios de ese grupo.

In [34]:
#DICCIONARIO DE CARACTERÍSTICAS DE LOS COMENTARIOS. 

comentarios_corpus = []
odio = []
no_odio = []

for d, v in zip(doc, value):
    comentario = {
        "texto": d.text,
        "num_palabras": sum(token.is_alpha for token in d),
        "num_oraciones": len(list(d.sents)),
        "num_entidades": len(d.ents),
        #Características adicionales
        "num_stopwords": sum(token.is_stop for token in d),
        "usa_mayusculas": any(token.is_upper and token.is_alpha for token in d),
        "num_verbos": sum(token.pos_ == "VERB" for token in d)
    }
    comentarios_corpus.append(comentario)

    if v > 0.0:  
        odio.append(comentario)
    elif v == 0.0:
        no_odio.append(comentario)

print("Numero de comentarios analizados:", len(comentarios_corpus))
print("Numero de comentarios de odio:", len(odio))
print("Numero de comentarios no de odio:", len(no_odio))

Numero de comentarios analizados: 10000
Numero de comentarios de odio: 332
Numero de comentarios no de odio: 9668


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 1.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuántos registros contiene el corpus?</span>

In [ ]:

print("Registros totales del corpus:", registros_total)
print("Comentarios en el corpus:", n_comentarios)
print("Comentarios analizados:", len(doc)) ##len(doc) == len(comentarios_corpus) == len(value) == lines_number


Registros totales del corpus: 575029
Comentarios en el corpus: 334318
Comentarios analizados: 10000


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Vamos a usar el corpus limitado a 10.000 para agilizar el tiempo de ejecucción del código. Pero realmente, el dataset completo tiene un corpus de 575029 registros.

Un punto importante es que si filtramos ya en primera instancia entre comentarios y no comentarios, es decir, titulares y otros tipos de textos que no nos hacen falta para nuestra tarea de análisis de odio, el corpus lo podemos reducir aún más, quedando en 334318 registros totales en el corpus.

Por lo tanto, yo he decidido filtrar en "Comentarios", y de ese set de 334.318 he escogido los 10.000 a analizar de forma aleatoria con una semilla para reproducibilidad.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 2.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuántas palabras totales hay en los comentarios del corpus?</span>

In [26]:
num_palabras_total = sum(
    c["num_palabras"]
    for c in comentarios_corpus
)

print("Número total de palabras:", num_palabras_total)

Número total de palabras: 455513


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

En el corpus que estamos trabajando de 10.000 entradas, tenemos un total de 455.513 palabras.
Como tenemos el diccionario con todos los comentarios del corpus junto con la información que necesitamos en cada pregunta solo hay que acceder al apartado "num_palabras" de cada comentario y hacer una suma total.
Este apartado hace uso de is_alpha para diferenciar si un token es una palabra o no (sólamente caracteres sin espacios ni signos).

Previamente , estaba haciendo esto pregunta a pregunta pero he decidido que es mucho más cómodo crear el diccionario de antemano e ir modificándolo según lo que necesitemos



<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 3.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Cuál el número promedio de palabras en cada comentario?</span>

In [27]:
num_palabras_por_comentario = [c["num_palabras"] for c in comentarios_corpus]
print("Promedio de palabras por comentario:", sum(num_palabras_por_comentario) / len(comentarios_corpus))

Promedio de palabras por comentario: 45.5513


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Para esta pregunta creamos la variable num_palabras_por_comentario para guardar una lista con el número total de palabras de cada comentario usando "num_palabras" ya definido en el diccionario. Tan sólo es hacer una división entre el total de palabras del corpus (455.513) entre el número total de comentarios del corpus. Como estamos usando el corpus limitado a 10.000 comentarios, el resultado es 455.513/10.000 = 45,5513 

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 4.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál el número promedio de palabras en los comentarios de cada grupo?</span>

In [28]:
num_palabras_odio = sum(c["num_palabras"]for c in odio)
num_palabras_no_odio = sum(c["num_palabras"]for c in no_odio)
print("Número total de palabras en comentarios de odio:", num_palabras_odio)
print("Número total de palabras en comentarios de no odio:", num_palabras_no_odio)
print("Promedio de palabras por comentario de odio:", num_palabras_odio / len(odio))
print("Promedio de palabras por comentario de no odio:", num_palabras_no_odio / len(no_odio))



Número total de palabras en comentarios de odio: 4838
Número total de palabras en comentarios de no odio: 450675
Promedio de palabras por comentario de odio: 14.572289156626505
Promedio de palabras por comentario de no odio: 46.615122052130744


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Primero sacamos la suma total de palabras de odio y de no odio, sumando el campo num_palabras de los diccionarios de cada grupo (sum(c["num_palabras"] for c in odio) y lo mismo para no_odio). Después dividimos cada total entre el número de comentarios de su grupo (len(odio) y len(no_odio)), y así obtenemos el promedio de palabras por comentario de cada grupo. El criterio de palabra es el mismo de la Pregunta 2 (is_alpha), de modo que la suma de ambos totales coincide con las 455.513 palabras del corpus analizado

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 5.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el número promedio de oraciones en los comentarios de cada grupo?</span>

In [32]:
num_oraciones_odio = sum(c["num_oraciones"]for c in odio)
num_oraciones_no_odio = sum(c["num_oraciones"]for c in no_odio)
print("Número total de oraciones en comentarios de odio:", num_oraciones_odio)
print("Número total de oraciones en comentarios de no odio:", num_oraciones_no_odio)
print("Promedio de oraciones por comentario de odio:", round(num_oraciones_odio / len(odio), 4))
print("Promedio de oraciones por comentario de no odio:", round(num_oraciones_no_odio / len(no_odio), 4))

Número total de oraciones en comentarios de odio: 526
Número total de oraciones en comentarios de no odio: 22112
Promedio de oraciones por comentario de odio: 1.5843
Promedio de oraciones por comentario de no odio: 2.2871


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Para este apartado he creado una nueva característica en el diccionario "num_oraciones": len(list(d.sents)). En spacy el .sents nos identifica las oraciones teniendo en cuenta signos de puntuación/cierre. Aquí hacemos que num_oraciones como lista guarde un número de oraciones por comentario.

Posteriormente, contabilizamos el numero de oraciones de cada grupo haciendo una suma de todos, y finalmente podemos hacer una división de la suma entre el número total de comentarios de cada grupo como en preguntas anteriores. 

Mostramos el promedio redondeando a 4 decimales para una presentación más limpia, pero guardamos en la variable el valor real sin truncar.
       

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Pregunta 6.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Considerando dos grupos de comentarios (odio y no odio) ¿Cuál es el porcentaje de comentarios que contienen entidades NER en cada grupo?</span>

In [109]:
con_ner_odio    = sum(1 for c in odio if c["num_entidades"] > 0)
con_ner_no_odio = sum(1 for c in no_odio if c["num_entidades"] > 0)
print("Odio:    ", con_ner_odio, "/", len(odio),
      "=", con_ner_odio / len(odio) * 100, "%")
print("No odio: ", con_ner_no_odio, "/", len(no_odio),
      "=", con_ner_no_odio / len(no_odio) * 100, "%")

Odio:     90 / 332 = 27.10843373493976 %
No odio:  3960 / 9668 = 40.95986760446835 %


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Ahora hemos añadido al diccionario la característica "num_entidades": len(d.ents), que nos dice para cada comentario el número de entidades NER que contiene haciendo uso de la opción .ents de spacy.
Para saber el porcentaje de comentarios que contienen entidades NER, debemos saber para cada comentario, si el "num_entidades" es mayor que 0. Entonces, hacemos para cada grupo un conteo de el número de comentarios que sí tienen entidad NER, para luego dividirla entre el número total de comentarios de cada grupo.

<span style="font-size: 16pt; font-weight: bold; color: #0098cd;">Plantea tus propias preguntas</span>

<span><b>Plantea al menos 4 características</b> del texto cuyo análisis permita una caracterización completa del texto. Puedes utilizar recomendaciones proporcionadas por la IA Generativa, si así lo deseas. Para cada una de las características planteadas, obtén valores separados para los grupos ODIO/NO-ODIO.</span>

<span>En la explicación aportada, deberás <b>explicar el significado de la característica planteada</b> así como la importancia de ésta en la caracterización del texto.</span>

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 1.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Proporción de stop words por grupo de comentarios (odio, no_odio)</span>

In [33]:
num_stopwords_odio = sum(c["num_stopwords"]for c in odio)
num_stopwords_no_odio = sum(c["num_stopwords"]for c in no_odio)
print("Número total de stopwords en comentarios de odio:", num_stopwords_odio)
print("Número total de stopwords en comentarios de no odio:", num_stopwords_no_odio)
print("Promedio de stopwords por comentario de odio:", round(num_stopwords_odio / len(odio), 4))
print("Promedio de stopwords por comentario de no odio:", round(num_stopwords_no_odio / len(no_odio), 4))


Número total de stopwords en comentarios de odio: 2750
Número total de stopwords en comentarios de no odio: 263454
Promedio de stopwords por comentario de odio: 8.2831
Promedio de stopwords por comentario de no odio: 27.2501


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

Para la primera característica adicional quería jugar con el término de stop words, y ver si podían tener alguna correlación con cada grupo.
Sorprendentemente, vemos que hay un promedio mucho mayor de stop words en comentarios de no odio. Esto, nos podría estar indicando que los de no odio tienen un lenguaje más rico, mas reflexivo y de debate y cuidando la gramática. Por otro lado podríamos decir que los comentarios de odio vienen dados por comportamientos impulsivos o de rabia en los que la persona que los pone no piensa detenidamente como está escribiendo.

El código fuente añadido consta de una nueva característica en nuestro diccionario del playground:

 "num_stopwords": sum(token.is_stop for token in d)

 Que nos indica simplemente la suma de el número de stop words totales en cada comentario. Por lo que con esa suma podemos sacar el promedio total dividiendo entre el número de comentarios de cada grupo.

 num_stopwords_odio = sum(c["num_stopwords"]for c in odio) recorre la característica num_stopwords que hemos definido, que devuelve un entero con el número de stop words detectadas por spacy en cada comentario y hace una suma de todos.




<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 2.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Porcentaje de uso de palabras en mayúscula por grupo de comentarios</span>

In [124]:
con_mayusculas_odio    = sum(1 for c in odio if c["usa_mayusculas"])
con_mayusculas_no_odio = sum(1 for c in no_odio if c["usa_mayusculas"])
porcentaje_mayusculas_odio = con_mayusculas_odio / len(odio) * 100
porcentaje_mayusculas_no_odio = con_mayusculas_no_odio / len(no_odio) * 100
print("Porcentaje de uso de palabras en mayúscula en comentarios de odio:", round(porcentaje_mayusculas_odio,4), "%")
print("Porcentaje de uso de palabras en mayúscula en comentarios de no odio:", round(porcentaje_mayusculas_no_odio,4), "%")


Porcentaje de uso de palabras en mayúscula en comentarios de odio: 0.3012 %
Porcentaje de uso de palabras en mayúscula en comentarios de no odio: 0.0517 %


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

Esta característica nos indica en un porcentaje la probabilidad en la que aparecen palabras con todas sus letras en mayúscula.
Me parece muy interesante, ya que podemos contabilizar si en comentarios de odio se usan más palabras mayúsculas para indicar cabreo o énfasis en el mensaje. De nuevo, correlacionado con la característica 1, en los mensajes de odio se nota la actitud agresiva del mensajero y las mayúsculas son una forma muy sencilla y visual de plasmar esto. Comprobamos que en odio es un 0.3% mientas que en los de no odio un 0.05%. Esta diferencia nos indica claramente que nuestra hipótesis es cierta

En el código añadimos a nuestro diccionario:

"usa_mayusculas": any(token.is_upper and token.is_alpha for token in d)

que comprueba si es una palabra y si todos sus carácteres están en mayúscula. y devuelve verdadero (1) si hay alguna palabra que cumple esos requisitos por cada comentario.

con_mayusculas_odio    = sum(1 for c in odio if c["usa_mayusculas"])
con_mayusculas_no_odio = sum(1 for c in no_odio if c["usa_mayusculas"])

Realiza una suma de conteo en las que por cada comentario suma 1 si usa_mayusculas ha encontrado alguna palabra en mayúsculas.


porcentaje_mayusculas_odio = con_mayusculas_odio / len(odio) * 100
porcentaje_mayusculas_no_odio = con_mayusculas_no_odio / len(no_odio) * 100

Finalmente, se hace una división de el número de comentarios que sí tienen alguna palabra en mayúscula entre el número de comentarios total por cada grupo, y se multiplica por cien para sacar el tanto por ciento.





<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 3.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Número y promedio de verbos por grupo de comentarios</span>

In [38]:
num_verbos_odio = sum(c["num_verbos"]for c in odio)
num_verbos_no_odio = sum(c["num_verbos"]for c in no_odio)
print("Número total de verbos en comentarios de odio:", num_verbos_odio)
print("Número total de verbos en comentarios de no odio:", num_verbos_no_odio)
print("Promedio de verbos por comentario de odio:", round(num_verbos_odio / len(odio), 4))
print("Promedio de verbos por comentario de no odio:", round(num_verbos_no_odio / len(no_odio), 4)) 

Número total de verbos en comentarios de odio: 641
Número total de verbos en comentarios de no odio: 51440
Promedio de verbos por comentario de odio: 1.9307
Promedio de verbos por comentario de no odio: 5.3206


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

Esta es muy simple: Primero añadimos la característica num_verbos al diccionario

"num_verbos": sum(token.pos_ == "VERB" for token in d)

Que para cada token analiza si es de tipo VERB (verbo), y hace una suma devolviendo el número total por comentario.
Luego, como en el resto, hacemos una división por grupo obteniendo el promedio de verbos en cada grupo odio y no odio.

La motivación de realizar esta característica es para acercarnos a un análisis mas profundo del texto y saber los límites de spacy. En la siguiente característica vamos mas allá.

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Característica adicional 4.</span>


<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Proporción de insultos por grupo de comentarios (odio, no odio)</span>

In [46]:
insultos = {
    "ahorcado", "ahorcamiento", "ahorcar", "amenaza", "amenazando",
    "amenazar", "amenazas", "amenazó", "analfabeta", "analfabetas",
    "analfabeto", "analfabetos", "anormal", "anormales", "asesina",
    "asesinada", "asesinadas", "asesinado", "asesinados", "asesinar",
    "asesinas", "asesinato", "asesinatos", "asesino", "asesinos",
    "asesinó", "asquerosa", "asquerosas", "asqueroso", "asquerosos",
    "basura", "basuras", "bocazas", "bomba", "bombas",
    "cabron", "cabrona", "cabronazos", "cabrones", "cabrón",
    "caradura", "caraduras", "carapolla", "cerda", "cerdas",
    "cerdo", "cerdos", "charlatan", "charlatanes", "charlatán",
    "chupoptero", "chupopteros", "chupóptero", "chupópteros", "cobarde",
    "cobardes", "cochina", "cochinas", "cochino", "cochinos",
    "colgado", "colgar", "comemierdas", "comunista", "comunistas",
    "comunistoide", "cono", "corrupta", "corruptas", "corrupto",
    "corruptos", "covidiota", "covidiotas", "coño", "cretina",
    "cretinas", "cretino", "cretinos", "culo", "culos",
    "degollado", "degollar", "delincuente", "delincuentes", "descerebrada",
    "descerebradas", "descerebrado", "descerebrados", "desgraciada", "desgraciadas",
    "desgraciado", "desgraciados", "disparar", "dispararon", "disparo",
    "disparos", "escoria", "estupida", "estupidas", "estupideces",
    "estupidez", "estupido", "estupidos", "estúpida", "estúpidas",
    "estúpido", "estúpidos", "explosivo", "explosivos", "exterminar",
    "exterminarlos", "facha", "fachas", "fachascal", "farsante",
    "farsantes", "fascista", "fascistas", "feminazi", "feminazis",
    "fusilado", "fusilamiento", "fusilar", "gentuza", "gilipolla",
    "gilipollas", "gilipolleces", "gilipollez", "giliprogres", "gilipuertas",
    "guarra", "guarras", "guarro", "guarros", "hdlgp",
    "hdp", "hijadeputa", "hijo de puta", "hijodeputa", "hijoputa",
    "hijosdeputa", "hijueputa", "hipocrita", "hipocritas", "hipócrita",
    "hipócritas", "hostia", "hostias", "hostion", "hostión",
    "hp", "idiota", "idiotas", "imbecil", "imbeciles",
    "imbecilidad", "imbécil", "imbéciles", "impresentable", "impresentables",
    "inepta", "ineptas", "inepto", "ineptos", "infraser",
    "inmundicia", "inutil", "inutiles", "inutilidad", "inutilidades",
    "inútil", "inútiles", "jeta", "jetas", "jilipollas",
    "jilipolleces", "jilipollez", "joda", "jodan", "jodas",
    "joder", "joderlos", "joderos", "joderse", "jodes",
    "jodete", "jodida", "jodidas", "jodido", "jodidos",
    "jodiendo", "jódete", "ladron", "ladrona", "ladronas",
    "ladrones", "ladrón", "lameculos", "maldita", "malditas",
    "maldito", "malditos", "maleducada", "maleducadas", "maleducado",
    "maleducados", "malnacida", "malnacidas", "malnacido", "malnacidos",
    "malparida", "malparidas", "malparido", "malparidos", "mamarracho",
    "mamarrachos", "manipulador", "manipuladora", "manipuladoras", "manipuladores",
    "marica", "maricon", "maricona", "mariconas", "maricones",
    "maricón", "mata", "matada", "matadas", "matado",
    "matados", "matamos", "matan", "matando", "matar",
    "matarla", "matarlas", "matarle", "matarlo", "matarlos",
    "matarme", "matarnos", "mataron", "mataros", "matarse",
    "matarte", "matas", "mato", "mató", "mentecata",
    "mentecatas", "mentecato", "mentecatos", "mentirosa", "mentirosas",
    "mentiroso", "mentirosos", "mequetrefe", "mequetrefes", "mhdp",
    "mierda", "mierdas", "miserable", "miserables", "mora",
    "moras", "morimos", "morir", "moriran", "moriras",
    "morirá", "morirán", "morirás", "moro", "moros",
    "morralla", "mueran", "mueras", "muere", "mueren",
    "muerete", "muérete", "negrata", "negratas", "okupa",
    "okupas", "ostia", "ostias", "paliza", "palizas",
    "parasito", "parasitos", "parásito", "parásitos", "payasa",
    "payasas", "payaso", "payasos", "pegado", "pegan",
    "pegando", "pegar", "pegarle", "pegarles", "pendeja",
    "pendejas", "pendejo", "pendejos", "pepero", "peperos",
    "perra", "perras", "perroflauta", "perroflautas", "piojosos",
    "podemita", "podemitas", "polla", "pollas", "pperro",
    "pperros", "progre", "progres", "progresista", "progresistas",
    "psicopata", "psicopatas", "psicópata", "psicópatas", "pudra",
    "pudran", "pudras", "pudrete", "pudrirse", "puta",
    "putas", "putisima", "puto", "putos", "quemado",
    "quemados", "quemando", "quemar", "rata", "ratas",
    "retrasada", "retrasadas", "retrasado", "retrasados", "retrasao",
    "retraso", "reventar", "roja", "rojas", "rojo",
    "rojos", "sanchista", "sanchistas", "sinverguenza", "sinverguenzas",
    "sinvergüenza", "sinvergüenzas", "socialcomunista", "socialcomunistas", "soplapollas",
    "soplapollez", "subnormal", "subnormales", "sudaca", "sudacas",
    "tarada", "taradas", "tarado", "tarados", "tiro",
    "tiros", "tonta", "tontas", "tonto", "tontolaba",
    "tontos", "traidor", "traidora", "traidoras", "traidores",
    "violacion", "violaciones", "violación", "violador", "violadora",
    "violadoras", "violadores", "violan", "violar", "zorra",
    "zorras"
}

In [47]:
resultados_insultos = []

for d, v in zip(doc, value):
    tokens = [t.lower_ for t in d if t.is_alpha]

    num_insultos = sum(
        palabra in insultos
        for palabra in tokens
    )

    resultados_insultos.append({
        "INTENSIDAD": v,
        "num_insultos": num_insultos,
        "tiene_insulto": num_insultos > 0
    })

df_insultos = pd.DataFrame(resultados_insultos)

In [48]:
df_insultos["grupo"] = df_insultos["INTENSIDAD"].apply(
    lambda v: "odio" if v > 0 else "no odio"
)

resumen = df_insultos.groupby("grupo").agg(
    total_insultos=("num_insultos", "sum"),
    promedio_insultos=("num_insultos", "mean"),
    comentarios_con_insulto=("tiene_insulto", "sum"),
    total_comentarios=("num_insultos", "size")
)

resumen["porcentaje_con_insulto"] = (
    resumen["comentarios_con_insulto"] / resumen["total_comentarios"] * 100
)

resumen = resumen.rename(columns={
    "total_insultos": "Total insultos",
    "promedio_insultos": "Promedio por comentario",
    "comentarios_con_insulto": "Comentarios con insulto",
    "total_comentarios": "Total comentarios",
    "porcentaje_con_insulto": "% con insulto",
})

tabla = resumen.round(4).style.format({
    "Total insultos": "{:.0f}",
    "Promedio por comentario": "{:.4f}",
    "Comentarios con insulto": "{:.0f}",
    "Total comentarios": "{:.0f}",
    "% con insulto": "{:.2f} %",
}).set_caption("Insultos por grupo (odio / no odio)")

display(tabla)


,Total insultos,Promedio por comentario,Comentarios con insulto,Total comentarios,% con insulto
grupo,,,,,
no odio,1754,0.1814,1372,9668,14.19 %
odio,270,0.8133,208,332,62.65 %


<b>Incluye aquí, debajo de la línea, la explicación de la característica propuesta y su motivación. Incluye también una explicación del código fuente aportado.</b>
<hr>
 

Para esta cuarta característica he construido un léxico de insultos y amenazas a partir de los términos que aparecen con más frecuencia en los comentarios de odio del corpus. No es una lista cerrada de palabras aisladas: incluye variantes con y sin tilde, singular y plural, formas coloquiales y compactas, insultos directos, despectivos políticos o étnicos y verbos o expresiones de amenaza.

La motivación es doble: por un lado, el insulto es una marca lingüística directa de hostilidad y permite comprobar si el odio se concentra en comentarios con léxico explícito; por otro, las amenazas aportan una señal de violencia que no siempre aparece como insulto. Al separar por grupos podemos comparar tanto la proporción de comentarios que contienen al menos un término del léxico como el promedio de términos por comentario.

En el código, "insultos" es el conjunto de términos. Recorremos doc y value con zip para mantener la correspondencia entre cada comentario ya procesado por spaCy y su INTENSIDAD. Para cada Doc obtenemos:

 tokens = [t.lower_ for t in d if t.is_alpha]
 
 es decir, solo palabras alfabéticas en minúsculas, y num_insultos suma 1 por cada token que está en insultos. tiene_insulto indica si el comentario contiene al menos uno. Con df_insultos construimos un DataFrame y creamos la columna grupo a parir tde la INTENSIDAD: odio si es mayor que 0 y no odio si es 0, los mismos grupos que venimos usando. Después, groupby("grupo").agg(...) calcula el total de insultos, el promedio por comentario, el número de comentarios con insulto y el total de comentarios; el porcentaje se obtiene dividiendo los comentarios con insulto entre el total de cada grupo y multiplicando por 100.

Los resultados muestran una diferencia muy clara: en odio hay 270 términos en 332 comentarios (0,8133 por comentario) y el 62,65 % de los comentarios contiene al menos uno; en no odio hay 1.754 términos en 9.668 comentarios (0,1814 por comentario) y solo el 14,19 % contiene alguno. Es decir, la presencia de insultos y amenazas es mucho más frecuente en el grupo de odio, aunque también aparece en no odio, probablemente por comentarios que citan o discuten insultos sin ser odio. Esto convierte la característica en una señal útil, pero no suficiente por sí sola.

<span style="font-size: 16pt; font-weight: bold; color: #0098cd;">Reflexión final</span>

<span>Una de las utilidades de la caracterización de texto es que nos sirve como etapa de <i>feature-extraction</i> (extración de características) de cara a un posterior sistema de clasificación. Es pertinente, por tanto, reflexionar sobre la capacidad discriminatoria de cada una de las características extraídas. </span>

<span> Responde, para ello, a la siguiente pregunta.</span>

<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">Reflexión final.</span>
<span style="font-size: 14pt; font-weight: bold; color: #0098cd;">¿Es posible utilizar alguna de las características extraídas en las preguntas anteriores para determinar si un mensaje contiene odio? Justifica tu respuesta con el análisis estadístico que consideres necesario.</span>

In [ ]:
from math import erfc, sqrt
from IPython.display import display

df_reflexion = pd.DataFrame([
    {
        "grupo": "odio" if v > 0 else "no odio",
        "num_palabras": c["num_palabras"],
        "num_oraciones": c["num_oraciones"],
        "num_entidades": c["num_entidades"],
        "num_stopwords": c["num_stopwords"],
        "num_verbos": c["num_verbos"],
        "usa_mayusculas": int(bool(c["usa_mayusculas"])),
    }
    for c, v in zip(comentarios_corpus, value)
])

df_reflexion["num_insultos"] = df_insultos["num_insultos"].values
df_reflexion["tiene_insulto"] = df_insultos["tiene_insulto"].astype(int).values

y = (df_reflexion["grupo"] == "odio").astype(int)
n1 = int(y.sum())
n0 = len(y) - n1

print(f"Odio: {n1} | No odio: {n0} | Prevalencia: {100 * n1 / len(y):.2f} %")
print(f"Exactitud de clasificar todo como no odio: {100 * n0 / len(y):.2f} %")
print("AUC útil = max(AUC, 1 - AUC): 0,5 es azar y 1 es separación perfecta.\n")

filas = []
for col in ["num_palabras", "num_oraciones", "num_entidades", "num_stopwords", "num_verbos", "num_insultos"]:
    a = df_reflexion.loc[y == 1, col]
    b = df_reflexion.loc[y == 0, col]
    ranks = df_reflexion[col].rank(method="average")
    auc = (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    filas.append({
        "caracteristica": col,
        "media_odio": a.mean(),
        "mediana_odio": a.median(),
        "media_no_odio": b.mean(),
        "mediana_no_odio": b.median(),
        "AUC": auc,
        "AUC_util": max(auc, 1 - auc),
    })

tabla_numericas = pd.DataFrame(filas)
tabla_numericas["caracteristica"] = tabla_numericas["caracteristica"].replace({
    "num_palabras": "Palabras",
    "num_oraciones": "Oraciones",
    "num_entidades": "Entidades",
    "num_stopwords": "Stopwords",
    "num_verbos": "Verbos",
    "num_insultos": "Insultos",
})
tabla_numericas = tabla_numericas.rename(columns={
    "caracteristica": "Característica",
    "media_odio": "Media odio",
    "mediana_odio": "Mediana odio",
    "media_no_odio": "Media no odio",
    "mediana_no_odio": "Mediana no odio",
    "AUC_util": "AUC útil",
})

print("CARACTERÍSTICAS NUMÉRICAS POR GRUPO")
display(tabla_numericas.round(3))

filas = []
for col in ["usa_mayusculas", "tiene_insulto"]:
    x = df_reflexion[col]
    tp = int(((y == 1) & (x == 1)).sum())
    fp = int(((y == 0) & (x == 1)).sum())
    fn = int(((y == 1) & (x == 0)).sum())
    tn = int(((y == 0) & (x == 0)).sum())

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    especificidad = tn / (tn + fp) if tn + fp else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    npv = tn / (tn + fn) if tn + fn else 0.0
    exactitud = (tp + tn) / len(y)
    odds = (tp * tn) / (fp * fn) if fp * fn else float("inf")
    den = (tp + fp) * (fn + tn) * (tp + fn) * (fp + tn)
    chi2 = len(y) * (tp * tn - fp * fn) ** 2 / den if den else 0.0
    p_valor = erfc(sqrt(chi2 / 2))
    auc = (recall + especificidad) / 2

    filas.append({
        "Característica": "Usa mayúsculas" if col == "usa_mayusculas" else "Contiene insulto",
        "Verdaderos positivos (TP)": f"{tp:,}".replace(",", "."),
        "Falsos positivos (FP)": f"{fp:,}".replace(",", "."),
        "Falsos negativos (FN)": f"{fn:,}".replace(",", "."),
        "Verdaderos negativos (TN)": f"{tn:,}".replace(",", "."),
        "Precisión": f"{100 * precision:.2f} %",
        "Recall (sensibilidad)": f"{100 * recall:.2f} %",
        "Especificidad": f"{100 * especificidad:.2f} %",
        "F1": f"{f1:.3f}",
        "NPV (valor predictivo negativo)": f"{100 * npv:.2f} %",
        "Exactitud": f"{100 * exactitud:.2f} %",
        "Odds ratio": f"{odds:.2f}",
        "Chi cuadrado": f"{chi2:.1f}",
        "p-valor": f"{p_valor:.3e}",
        "AUC": f"{auc:.3f}",
    })

tabla_binarias = pd.DataFrame(filas).set_index("Característica").T

print("\nCARACTERÍSTICAS BINARIAS: MÉTRICAS DE CLASIFICACIÓN")
display(tabla_binarias)

Odio: 332 | No odio: 9668 | Prevalencia: 3.32 %
Exactitud de clasificar todo como no odio: 96.68 %
AUC útil = max(AUC, 1 - AUC): 0,5 es azar y 1 es separación perfecta.

CARACTERÍSTICAS NUMÉRICAS POR GRUPO


,Característica,Media odio,Mediana odio,Media no odio,Mediana no odio,AUC,AUC útil
0,Palabras,14.572,9.0,46.615,18.0,0.320,0.680
1,Oraciones,1.584,1.0,2.287,2.0,0.412,0.588
2,Entidades,0.349,0.0,1.268,0.0,0.417,0.583
3,Stopwords,8.283,4.5,27.250,11.0,0.311,0.689
4,Verbos,1.931,1.0,5.321,2.0,0.346,0.654
5,Insultos,0.813,1.0,0.181,0.0,0.742,0.742



CARACTERÍSTICAS BINARIAS: MÉTRICAS DE CLASIFICACIÓN


Característica,Usa mayúsculas,Contiene insulto
Verdaderos positivos (TP),1,208
Falsos positivos (FP),5,1.372
Falsos negativos (FN),331,124
Verdaderos negativos (TN),9.663,8.296
Precisión,16.67 %,13.16 %
Recall (sensibilidad),0.30 %,62.65 %
Especificidad,99.95 %,85.81 %
F1,0.006,0.218
NPV (valor predictivo negativo),96.69 %,98.53 %
Exactitud,96.64 %,85.04 %


<b>Incluye aquí, debajo de la línea, la explicación de tu respuesta</b>
<hr>
 

Para interpretar estos resultados hay que tener en cuenta, primero, el desbalance de clases: solo 332 de los 10.000 comentarios son de odio (3,32 %) frente a 9.668 de no odio (96,68 %). Por eso la exactitud (accuracy) no es una buena referencia: un clasificador que dijera siempre "no odio" acertaría el 96,68 % sin detectar ni un solo caso de odio. Hay que fijarse en el recall, la precisión, el F1, el NPV y el AUC.

Características numéricas. Los comentarios de odio son, de media, mucho más cortos (14,57 palabras frente a 46,62) y tienen menos oraciones (1,58 frente a 2,29), menos entidades (0,35 frente a 1,27), menos stopwords (8,28 frente a 27,25) y menos verbos (1,93 frente a 5,32). En cambio, contienen más insultos (0,81 frente a 0,18). Las medianas confirman la misma tendencia y son más fiables que las medias, porque el grupo de no odio tiene valores muy extremos (por ejemplo, la desviación típica de num_palabras es 112,58). La lectura es que el odio tiende a aparecer en mensajes breves, directos y con menos elaboración sintáctica, mientras que los comentarios de no odio son más largos y desarrollados.

El AUC mide la probabilidad de que un comentario de odio tomado al azar tenga un valor mayor que uno de no odio. 0,5 es azar y 1 sería separación perfecta. En num_palabras, num_stopwords, num_verbos, num_oraciones y num_entidades el AUC bruto sale por debajo de 0,5 porque la relación es inversa (el odio tiene valores más bajos). Invirtiendo el sentido, el AUC útil es 0,680, 0,689, 0,654, 0,588 y 0,583 respectivamente. Es decir, la longitud y las stopwords discriminan de forma moderada y las oraciones y entidades, de forma débil. La mejor característica numérica es num_insultos, con AUC 0,742.

Características binarias. En tiene_insulto el recall es 62,65 %: detecta 208 de los 332 comentarios de odio. La precisión es 13,16 % ya que de los 1.580 comentarios marcados, solo 208 son realmente de odio. La especificidad es 85,81 % y el NPV es 98,53 %. Esto quiere decir que si un comentario no contiene insultos, hay un 98,53 % de probabilidad de que no sea odio. El F1 es 0,218 y el p-valor es aproximadamente 3,11e-125, con AUC 0,742. Esto indica una asociación muy fuerte y estadísticamente significativa: el insulto es una señal útil, sobre todo para descartar odio, pero no para confirmarlo, porque muchos comentarios de no odio también insultan (crítica, ironía, citas, etc.).

En usa_mayusculas solo hay 1 verdadero positivo y 5 falsos positivos (TP=1, FP=5, FN=331, TN=9.663). El recall es 0,3 %, la precisión 16,67 %, el F1 0,006 y el p-valor 0,068. Aunque la proporción sea mayor en odio, se apoya en muy pocos casos y no alcanza significación estadística así que en esta muestra no es una característica fiable por sí sola.

Conclusión. Sí es posible utilizar estas características como predictoras, pero ninguna determina el odio con certeza. La más discriminativa es tiene_insulto que es útil como cribado o regla de descarte. Las numéricas aportan una señal débil o moderada, y usa_mayusculas no es concluyente. Para un sistema real habría que combinar varias características, validar con un conjunto de prueba y ajustar el umbral, evaluando con precisión, recall, F1 y AUC en lugar de exactitud. Además, el odio puede ser implícito y no contener insultos, y un insulto puede aparecer en un comentario que no sea de odio, por lo que el contexto sigue siendo imprescindible.